In [ ]:
import asyncio, httpx, time

**Use async HTTPX client**

In [13]:
async def get_data(i: int, endpoint: str):
    start = time.time()
    print(f"Start runnning {endpoint}: {i}")
    url = f"http://127.0.0.1:8000/{endpoint}/{i}"

    async with httpx.AsyncClient() as client:
        r = await client.get(url)

    print(f"Finisher running {endpoint}: {i} in {time.time() - start:.2f}s")

In [14]:
await get_data(1, "sync")  # instead of asyncio.run()

Start runnning sync: 1
Finisher running sync: 1 in 3.21s


In [15]:
await get_data(1, "async")  # instead of asyncio.run()

Start runnning async: 1
Finisher running async: 1 in 3.23s


**Attacking Endpoints** - serve multiple requests concurrently

In [28]:
# sync endpoint
start = time.time()
await asyncio.gather(
    *[get_data(i, "sync") for i in range(300)]
)
print(f"Finished running ALL in {time.time() - start:.2f}s")

Start runnning sync: 0
Start runnning sync: 1
Start runnning sync: 2
Start runnning sync: 3
Start runnning sync: 4
Start runnning sync: 5
Start runnning sync: 6
Start runnning sync: 7
Start runnning sync: 8
Start runnning sync: 9
Start runnning sync: 10
Start runnning sync: 11
Start runnning sync: 12
Start runnning sync: 13
Start runnning sync: 14
Start runnning sync: 15
Start runnning sync: 16
Start runnning sync: 17
Start runnning sync: 18
Start runnning sync: 19
Start runnning sync: 20
Start runnning sync: 21
Start runnning sync: 22
Start runnning sync: 23
Start runnning sync: 24
Start runnning sync: 25
Start runnning sync: 26
Start runnning sync: 27
Start runnning sync: 28
Start runnning sync: 29
Start runnning sync: 30
Start runnning sync: 31
Start runnning sync: 32
Start runnning sync: 33
Start runnning sync: 34
Start runnning sync: 35
Start runnning sync: 36
Start runnning sync: 37
Start runnning sync: 38
Start runnning sync: 39
Start runnning sync: 40
Start runnning sync: 41
St

ReadTimeout: 

In [26]:
# async endpoint
start = time.time()
await asyncio.gather(
    *[get_data(i, "async") for i in range(30)]
)
print(f"Finished running ALL in {time.time() - start:.2f}s")

Start runnning async: 0
Start runnning async: 1
Start runnning async: 2
Start runnning async: 3
Start runnning async: 4
Start runnning async: 5
Start runnning async: 6
Start runnning async: 7
Start runnning async: 8
Start runnning async: 9
Start runnning async: 10
Start runnning async: 11
Start runnning async: 12
Start runnning async: 13
Start runnning async: 14
Start runnning async: 15
Start runnning async: 16
Start runnning async: 17
Start runnning async: 18
Start runnning async: 19
Start runnning async: 20
Start runnning async: 21
Start runnning async: 22
Start runnning async: 23
Start runnning async: 24
Start runnning async: 25
Start runnning async: 26
Start runnning async: 27
Start runnning async: 28
Start runnning async: 29
Finisher running async: 21 in 5.05s
Finisher running async: 29 in 3.26s
Finisher running async: 20 in 5.26s
Finisher running async: 25 in 4.07s
Finisher running async: 18 in 5.66s
Finisher running async: 0 in 9.31s
Finisher running async: 17 in 5.86s
Finisher 

**Using `aiohttp` client**

In [5]:
import aiohttp, time, asyncio


async def get_data(i: int, endpoint: str):
    start = time.time()
    print(f"Start runnning {endpoint}: {i}")
    url = f"http://127.0.0.1:8000/{endpoint}/{i}"

    async with aiohttp.ClientSession() as session:
        async with session.get(url) as r:

            print(f"Finisher running {endpoint}: {i} in {time.time() - start:.2f}s")

**Sync endpoints block the event loop, so Starlette/Uvicorn offload them to a threadpool**
```shell
ThreadPoolExecutor(max_workers=40)
```
This is the default.

So:
- each sync request → occupies one thread
- up to 40 threads → concurrency limit
- the 41st request waits
- the 300th request waits a long time

This is why sync endpoint takes 24 seconds for 300 requests.

In [ ]:
# sync endpoint
start = time.time()
await asyncio.gather(
    *[get_data(i, "sync") for i in range(3)]
)
print(f"Finished running ALL in {time.time() - start:.2f}s")

Start runnning sync: 0
Start runnning sync: 1
Start runnning sync: 2
Finisher running sync: 0 in 3.01s
Finisher running sync: 2 in 3.01s
Finisher running sync: 1 in 3.01s
Finished running ALL in 3.01s


**Async endpoints never spawn threads for I/O.**

**They run entirely inside the event loop**.

**async endpoints do NOT use threads for request handling**

They run inside:
- 1 Uvicorn worker
- 1 asyncio event loop
- 1 thread (the main loop thread)

4 threads are NOT request handlers:
- the main event loop thread
- uvicorn’s internal housekeeping threads
- logging thread
- watchdog thread

300 async requests → 1 event loop thread handles all of them concurrently.

**Why async endpoint handles 300 requests in ~3 seconds**

Because:
- await asyncio.sleep(3) yields control
- event loop schedules all 300 tasks concurrently
- no threads are used
- no blocking occurs

In [12]:
# async endpoint
start = time.time()
await asyncio.gather(
    *[get_data(i, "async") for i in range(300)]
)
print(f"Finished running ALL in {time.time() - start:.2f}s")

Start runnning async: 0
Start runnning async: 1
Start runnning async: 2
Start runnning async: 3
Start runnning async: 4
Start runnning async: 5
Start runnning async: 6
Start runnning async: 7
Start runnning async: 8
Start runnning async: 9
Start runnning async: 10
Start runnning async: 11
Start runnning async: 12
Start runnning async: 13
Start runnning async: 14
Start runnning async: 15
Start runnning async: 16
Start runnning async: 17
Start runnning async: 18
Start runnning async: 19
Start runnning async: 20
Start runnning async: 21
Start runnning async: 22
Start runnning async: 23
Start runnning async: 24
Start runnning async: 25
Start runnning async: 26
Start runnning async: 27
Start runnning async: 28
Start runnning async: 29
Start runnning async: 30
Start runnning async: 31
Start runnning async: 32
Start runnning async: 33
Start runnning async: 34
Start runnning async: 35
Start runnning async: 36
Start runnning async: 37
Start runnning async: 38
Start runnning async: 39
Start runn

**Why aiohttp is MUCH faster than httpx for high concurrency**

**`httpx.AsyncClient` is slower because**:
- it uses **httpcore** with Python‑level async
- it has more layers (middleware, dispatchers, adapters)
- it is optimized for correctness, not raw throughput
- it uses pure‑Python connection pooling
- it has higher overhead per request

**`aiohttp` is faster because**:
- it is built on libuv‑style C extensions
- it uses highly optimized C parsers
- it has extremely efficient connection pooling
- it is designed for high‑throughput async workloads

So:

> `aiohttp` can easily push 300 requests in 3 seconds

> `httpx` struggles because it saturates the Python event loop

This is normal and expected.

# Why sync endpoint takes ~24 seconds

Because:

- threadpool max = 40
- 300 requests → 40 run at once
- 260 wait in queue
- each batch takes 3 seconds
- 300 / 40 = 7.5 batches
- 7.5 × 3s ≈ 22.5 seconds
- - overhead ≈ 24 seconds

This is exactly what you observed.

# ⭐ 7. Summary table (this explains everything)

|Endpoint type|Concurrency model|Threads used|300 requests time|Why|
|---|---|---|---|---|
|**async**|event loop|1|~3 sec|all requests run concurrently|
|**sync**|threadpool|up to 40|~24 sec|threadpool saturation|
|**httpx async client**|Python async|1|slow|high overhead|
|**aiohttp async client**|C‑optimized async|1|fast|low overhead|


**`threading.active_count()`**

inside an **async** FastAPI endpoint, the number you see **does NOT represent “threads used for handling async requests.”** It includes:

- the main event‑loop thread
- uvicorn internal threads
- watchdog threads
- logging threads
- asyncio executor threads (for DNS, SSL, etc.)
- OS-level background threads Python starts automatically

So yes — the “4 threads” you see for async endpoints are **just housekeeping threads**, not request handlers.